# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, following the Croissant schema standard. All exploration refers to record sets and fields by their `@id` values, ensuring consistent and precise referencing across the dataset.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
In this section, we will load the metadata and records from the FAIR² dataset using `mlcroissant`.

Note: When referencing dataset structure, we use `@id` fields per the Croissant specification.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a summary
print("\u001b[1m{}\u001b[0m".format(metadata.name))
print(metadata.description)

## 2. Data Overview
We review the available record sets within the dataset, along with their `@id`s and associated fields. This information will guide the extraction and analysis steps.

**Note:** `mlcroissant` provides access to the dataset structure, including record sets, fields, and file objects.

Let's enumerate all record sets by `@id` and scan their fields.

In [ ]:
# List all record sets in the dataset (by @id and name)
record_sets = list(dataset.record_sets)
print("Available record sets (by @id):")
for rs in record_sets:
    print(f"  - @id: {rs.id} | name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - @id: {field.id} | name: {field.name} | dtype: {field.data_type}")
    print()

# For exploration, print a sample record (if available)
if record_sets:
    sample_record_set = record_sets[0]
    print(f"\nSample records from record set '@id': {sample_record_set.id}")
    for i, rec in enumerate(dataset.records(record_set=sample_record_set.id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
We load data from all available record sets into Pandas DataFrames for further analysis. All record sets are referenced by their `@id` values.

Fields and columns will be accessible using these `@id`s.

In [ ]:
# Extract data from each record set found above
dataframes = {}

for rs in record_sets:
    recs = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(recs)
    dataframes[rs.id] = df
    print(f"Loaded {len(df)} records for record set '@id': {rs.id}")
    if not df.empty:
        print(f"  Columns: {list(df.columns)}\n")

# Display the first few rows of the first record set as a sample
main_record_set_id = record_sets[0].id if record_sets else None
if main_record_set_id and not dataframes[main_record_set_id].empty:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process data from one of the main record sets. We'll:

- Select a numeric field by its `@id`
- Filter records based on a threshold
- Normalize the selected field
- Optionally, group by another field (using its exact `@id`)

Adjust the variable names to match the actual `@id`s.

In [ ]:
# Example: EDA on the main record set

import numpy as np

# Identify a record set and a sample numeric field (by @id):
if record_sets:
    rs = record_sets[0]
    df = dataframes[rs.id]
    # Find a numeric field (@id) to use
    numeric_fields = [f for f in rs.fields if f.data_type in ['Float', 'Number', 'Integer']]
    if numeric_fields:
        numeric_field = numeric_fields[0].id
        print(f"Using numeric field '@id': {numeric_field}")
        # filter rows where value > threshold (using mean if no clear threshold)
        if numeric_field in df.columns and not df.empty and pd.api.types.is_numeric_dtype(df[numeric_field]):
            threshold = df[numeric_field].mean()
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
            print(filtered_df.head())

            # Normalize...
            mean = filtered_df[numeric_field].mean()
            std = filtered_df[numeric_field].std()
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
            print(f"\nNormalized values for '{numeric_field}':")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Group by another (categorical) field if available
            candidate_group_fields = [f.id for f in rs.fields if f.data_type == 'Text' and f.id in df.columns]
            if candidate_group_fields:
                group_field = candidate_group_fields[0]
                print(f"\nGrouping by field '@id': {group_field}")
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(grouped_df.head())
        else:
            print("No numeric field available in the main record set or no data loaded.")
    else:
        print("No numeric fields identified in the main record set.")

## 5. Visualization
Let's visualize a numeric distribution from the main record set, if available, referencing all fields and record sets by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution, if data is present
if record_sets and numeric_fields and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set '@id': {rs.id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing a Croissant-compliant dataset using `mlcroissant`, referencing all entities by their `@id`. We:
- Identified record sets and fields via their `@id`
- Loaded data into DataFrames
- Applied EDA and transformations using `@id` for all field references
- Visualized key attributes

**Continue exploration by selecting further record sets or applying detailed modeling to fields of interest using their precise `@id` references.**